# Metric inspection tool

This notebook inspects the training metrics across a list of experiment configuration files.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from flax import nnx
import pathlib
from tqdm import tqdm
from qnrl.tools import MetricStatistics, compute_train_result_statistics
from qnrl.cli import load_algo_config

In [3]:
# Define config files to load.
config_paths = [
    "../experiments/c51_CartPole-v1_config-0.py",
    "../experiments/qnrl_CartPole-v1_config-0.py",
    "../experiments/qnrl_CartPole-v1_config-1.py",
    "../experiments/qnrl_CartPole-v1_config-2.py",
    "../experiments/qnrl_CartPole-v1_config-3.py",
    "../experiments/qnrl_CartPole-v1_config-4.py",
    "../experiments/qnrl_CartPole-v1_config-5.py",
    "../experiments/qnrl_CartPole-v1_config-6.py",
    "../experiments/qnrl_CartPole-v1_config-7.py",
    "../experiments/qnrl_CartPole-v1_config-8.py",
    "../experiments/qnrl_CartPole-v1_config-9.py",
]
config_paths = [pathlib.Path(path) for path in config_paths]

configs = {cfg_path.stem: load_algo_config(path=cfg_path) for cfg_path in config_paths}

In [4]:
# Load metrics from designated config files.
data = {}
for cfg_key, cfg in tqdm(configs.items(), total=len(configs)):
    # Load metrics into memory.
    with cfg.metric_manager as metric_manager:
        metric_manager.load()
        metrics = metric_manager.get_metrics()
        # Collect metric information.
        data[cfg_key] = {m.key: m for m in compute_train_result_statistics(metrics)}

100%|██████████| 11/11 [00:12<00:00,  1.16s/it]


In [5]:
nnx.display(data)

In [6]:
def sort_by_metric(data: dict[str, dict[str, MetricStatistics]], metric_key: str, statistic_key: str, **kwargs):
    """Sort a dictionary of metrics across multiple configurations based on a specific metric and a statistic (e.g., min, max, mean, etc.).
    
    Example sort based on lowest mean loss (ascending order):
    >>> sort_by_metric(data, metric_key='loss', statistic_key='mean')
    """
    return sorted(list((cfg_key, metric_dict[metric_key].to_dict()[statistic_key]) for cfg_key, metric_dict in data.items()), key=lambda tup: tup[-1], **kwargs)

In [7]:
sort_by_metric(data, metric_key='loss', statistic_key='mean')

[('c51_CartPole-v1_config-0', 1.2098232183127327),
 ('qnrl_CartPole-v1_config-8', 2.545324289331176),
 ('qnrl_CartPole-v1_config-7', 2.6917699644628077),
 ('qnrl_CartPole-v1_config-6', 2.7469395415027233),
 ('qnrl_CartPole-v1_config-9', 2.8544370479207193),
 ('qnrl_CartPole-v1_config-1', 3.068294982288641),
 ('qnrl_CartPole-v1_config-4', 3.085224007937471),
 ('qnrl_CartPole-v1_config-5', 3.1843082027775513),
 ('qnrl_CartPole-v1_config-2', 3.3149442987427724),
 ('qnrl_CartPole-v1_config-0', 3.4246882126715033),
 ('qnrl_CartPole-v1_config-3', 3.494879278598099)]

In [8]:
sort_by_metric(data, metric_key='loss', statistic_key='min')

[('c51_CartPole-v1_config-0', 0.06410524595324565),
 ('qnrl_CartPole-v1_config-4', 1.5977780188840291),
 ('qnrl_CartPole-v1_config-7', 1.671187201208335),
 ('qnrl_CartPole-v1_config-5', 1.7190766148313295),
 ('qnrl_CartPole-v1_config-1', 1.8938576270390988),
 ('qnrl_CartPole-v1_config-2', 1.9240688243801984),
 ('qnrl_CartPole-v1_config-8', 1.9432444841208927),
 ('qnrl_CartPole-v1_config-6', 1.970978291581291),
 ('qnrl_CartPole-v1_config-0', 2.22519203983689),
 ('qnrl_CartPole-v1_config-9', 2.3349383515260627),
 ('qnrl_CartPole-v1_config-3', 2.622097361682192)]

In [9]:
sort_by_metric(data, metric_key='episode_reward', statistic_key='mean', reverse=True)

[('c51_CartPole-v1_config-0', 121.5660701152243),
 ('qnrl_CartPole-v1_config-9', 24.013507429085998),
 ('qnrl_CartPole-v1_config-7', 22.403622917056733),
 ('qnrl_CartPole-v1_config-6', 20.80937840785169),
 ('qnrl_CartPole-v1_config-4', 17.549022978625867),
 ('qnrl_CartPole-v1_config-5', 16.963615023474176),
 ('qnrl_CartPole-v1_config-0', 16.72948467058056),
 ('qnrl_CartPole-v1_config-3', 16.16079893475366),
 ('qnrl_CartPole-v1_config-8', 15.844057404662443),
 ('qnrl_CartPole-v1_config-2', 14.640859387878454),
 ('qnrl_CartPole-v1_config-1', 13.926018750580154)]

In [10]:
sort_by_metric(data, metric_key='episode_reward', statistic_key='max', reverse=True)

[('c51_CartPole-v1_config-0', 500.0),
 ('qnrl_CartPole-v1_config-4', 500.0),
 ('qnrl_CartPole-v1_config-5', 500.0),
 ('qnrl_CartPole-v1_config-1', 498.0),
 ('qnrl_CartPole-v1_config-9', 492.0),
 ('qnrl_CartPole-v1_config-3', 486.0),
 ('qnrl_CartPole-v1_config-6', 471.0),
 ('qnrl_CartPole-v1_config-8', 315.0),
 ('qnrl_CartPole-v1_config-0', 264.0),
 ('qnrl_CartPole-v1_config-2', 160.0),
 ('qnrl_CartPole-v1_config-7', 130.0)]